# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide to load, explore, and process the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant JSON-LD URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset via Croissant
dataset = mlc.Dataset(croissant_url)

# Grab a description using the Croissant Dataset object's .metadata
meta = dataset.metadata
print(f"Dataset name: {meta.name}\nDescription: {meta.description}")

## 2. Data Overview
Review available record sets and their fields. All references use each entity's Croissant schema `@id`. This ensures consistency when loading, exploring, or processing.

In [ ]:
# Print the available record sets and their fields using their @id

print("Record sets found:")
for record_set in dataset.metadata.record_sets:
    print(f"- {record_set.name} (@id: {record_set.id})")
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - {field.name} (@id: {field.id})")

## 3. Data Extraction
Load data from the primary record set into a pandas DataFrame. Fields and columns are referenced by their Croissant `@id`.

**Note**: For this dataset, the record set with the protein table is the main focus. We'll extract data from the `cr:RecordSet/ProteinTable` (replace with the actual ID if it differs).

In [ ]:
# Find the protein table record set by name or ID (adjust if needed)

# Gather all available record set ids
record_sets = [rs.id for rs in dataset.metadata.record_sets]
print("Record set @ids:", record_sets)

# We'll use the first record set (primary data table) for exploratory analysis
protein_table_id = record_sets[0]

# Load records into a DataFrame
records = list(dataset.records(record_set=protein_table_id))
df = pd.DataFrame(records)
print(f"Fields/Columns in the record set '{protein_table_id}':\n", df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Let's process numeric data: filter records, normalize a numeric column, and optionally group by a selected field. All columns used are referenced by their `@id`.

In [ ]:
# Identify a numeric field for analysis by checking types in the schema or data
# Example: Let's pick 'Coverage(%)' and 'MW' if available (find actual field @id)
numeric_candidates = [col for col in df.columns if 'coverage' in col.lower() or col.lower() in ['mw', 'molecular_weight', 'coverage', 'coverage_percent']]
print("Numeric candidates: ", numeric_candidates)
if numeric_candidates:
    numeric_field = numeric_candidates[0]
else:
    numeric_field = df.select_dtypes(include='number').columns[0]

# Display value counts and simple stats for basic EDA
print(f"Statistics for field: {numeric_field}")
print(df[numeric_field].describe())

# Filtering (e.g., proteins with MW > 50 kDa or coverage > 10)
threshold = df[numeric_field].mean()  # Or a constant like 10/50000 as appropriate
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean): {len(filtered_df)} records")
print(filtered_df.head(3))

# Normalize the numeric field (z-score normalization)
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Optionally, group by a categorical field (e.g., 'Sample' if available)
cat_candidates = [col for col in df.columns if col not in [numeric_field] and df[col].dtype == 'object']
group_field = cat_candidates[0] if cat_candidates else None
if group_field:
    print(f"Grouping by categorical field: {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
    print(grouped_df.head(3))

## 5. Visualization
Visualize the distribution of the selected numeric field and/or relationships between two variables.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.xlabel(numeric_field)
plt.title(f"Distribution of {numeric_field}")
plt.show()

# If there is a group field, show boxplots
if group_field:
    plt.figure(figsize=(12,5))
    sns.boxplot(data=df, x=group_field, y=numeric_field)
    plt.xticks(rotation=45)
    plt.title(f"{numeric_field} by {group_field}")
    plt.show()

## 6. Conclusion
This notebook demonstrated how to use `mlcroissant` to access, filter, and process the FAIR^2 dataset of mass spectrometry analysis results for human mast cell extracellular vesicles. By using Croissant `@id` identifiers for all entities, field selection and analysis remains robust even as the schema evolves. You can now build on these steps to perform deeper bioinformatics or statistical analyses as needed.